# PERSUADE Score Long - Curve Shape Metrics (v4)

Post-processing analysis of curve shape from existing ppl_W* columns.

**New Metrics:**
1. **AUC**: Area under the Δppl curve (trapezoidal integration)
2. **log_slope**: Slope of Δppl vs log(window) - measures decay rate
3. **ratio_128_32**: Δ128/Δ32 - early saturation indicator
4. **ratio_512_128**: Δ512/Δ128 - late benefit indicator
5. **slope_early**: Piecewise slope 32→128 (early decay)
6. **slope_late**: Piecewise slope 128→512 (late decay)

**Input:** `essay_level_results.csv` from score_long run (v3)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import trapezoid
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path

In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = Path("/content/drive/MyDrive/LRTIA/Results/Persuade/score_long")
except:
    # Local path
    RESULTS_DIR = Path("../results/persuade/score_long")

print(f"Results dir: {RESULTS_DIR}")

In [ ]:
# Load existing results
df = pd.read_csv(RESULTS_DIR / 'essay_level_results.csv')
print(f"Loaded {len(df)} essays")
print(f"\nScore bin distribution:")
print(df['score_bin'].value_counts())

# Windows
WINDOWS = [32, 64, 128, 256, 384, 512]
GROUP_ORDER = ['low', 'mid', 'high']
COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

## 1. Compute Curve Shape Metrics

In [ ]:
def compute_curve_metrics(row):
    """Compute curve shape metrics from ppl_W* columns."""
    
    # Extract perplexity values
    ppls = {W: row[f'ppl_W{W}'] for W in WINDOWS}
    
    # Check for NaN
    if any(pd.isna(v) for v in ppls.values()):
        return pd.Series({
            'auc': np.nan,
            'log_slope': np.nan,
            'ratio_128_32': np.nan,
            'ratio_512_128': np.nan,
            'slope_early': np.nan,
            'slope_late': np.nan,
        })
    
    # Convert to arrays
    windows = np.array(WINDOWS)
    ppl_vals = np.array([ppls[W] for W in WINDOWS])
    
    # Δppl from baseline (W=32)
    delta_ppl = ppl_vals[0] - ppl_vals  # Positive = benefit
    
    # ============================================================
    # 1. AUC: Area under Δppl curve (trapezoidal)
    # ============================================================
    # Integrate delta_ppl over windows
    auc = trapezoid(delta_ppl, windows)
    
    # ============================================================
    # 2. log_slope: Slope of Δppl vs log(window)
    # ============================================================
    log_windows = np.log(windows)
    # Linear regression: delta_ppl = a + b * log(window)
    slope, intercept, r, p, se = stats.linregress(log_windows, delta_ppl)
    log_slope = slope
    
    # ============================================================
    # 3. Ratios: Δ128/Δ32 and Δ512/Δ128
    # ============================================================
    # Δ32 = ppl_32 - ppl_64 (benefit of going from 32 to 64)
    # Δ128 = ppl_64 - ppl_128 (benefit of going from 64 to 128)
    # Actually, let's define as cumulative benefit:
    # Δ_to_64 = ppl_32 - ppl_64
    # Δ_to_128 = ppl_32 - ppl_128
    # Δ_to_512 = ppl_32 - ppl_512
    
    delta_to_64 = ppls[32] - ppls[64]
    delta_to_128 = ppls[32] - ppls[128]
    delta_to_512 = ppls[32] - ppls[512]
    
    # Ratio: how much of the 128 benefit came by 64?
    ratio_128_32 = delta_to_64 / delta_to_128 if delta_to_128 > 0.001 else np.nan
    
    # Ratio: late benefit (128→512) relative to early benefit (32→128)
    delta_late = ppls[128] - ppls[512]  # Benefit from 128 to 512
    ratio_512_128 = delta_late / delta_to_128 if delta_to_128 > 0.001 else np.nan
    
    # ============================================================
    # 4. Piecewise slopes
    # ============================================================
    # Early slope: 32→128 (per log-token)
    early_windows = np.array([32, 64, 128])
    early_ppls = np.array([ppls[32], ppls[64], ppls[128]])
    early_delta = ppls[32] - early_ppls
    slope_early, _, _, _, _ = stats.linregress(np.log(early_windows), early_delta)
    
    # Late slope: 128→512 (per log-token)
    late_windows = np.array([128, 256, 384, 512])
    late_ppls = np.array([ppls[128], ppls[256], ppls[384], ppls[512]])
    late_delta = ppls[128] - late_ppls  # Benefit relative to 128
    slope_late, _, _, _, _ = stats.linregress(np.log(late_windows), late_delta)
    
    return pd.Series({
        'auc': auc,
        'log_slope': log_slope,
        'ratio_128_32': ratio_128_32,
        'ratio_512_128': ratio_512_128,
        'slope_early': slope_early,
        'slope_late': slope_late,
    })

# Apply to all rows
metrics = df.apply(compute_curve_metrics, axis=1)
df = pd.concat([df, metrics], axis=1)

print("New metrics computed:")
print(df[['essay_id', 'score_bin', 'auc', 'log_slope', 'ratio_128_32', 'ratio_512_128', 'slope_early', 'slope_late']].head(10))

In [ ]:
# Descriptive stats for new metrics
NEW_METRICS = ['auc', 'log_slope', 'ratio_128_32', 'ratio_512_128', 'slope_early', 'slope_late']

print("="*80)
print("DESCRIPTIVE STATISTICS - NEW METRICS")
print("="*80)

for metric in NEW_METRICS:
    print(f"\n--- {metric} ---")
    print(f"{'score_bin':<10} {'n':>6} {'mean':>12} {'std':>12} {'median':>12}")
    print("-"*55)
    for group in GROUP_ORDER:
        g_df = df[df['score_bin'] == group]
        vals = g_df[metric].dropna()
        print(f"{group:<10} {len(vals):>6} {vals.mean():>12.3f} {vals.std():>12.3f} {vals.median():>12.3f}")

## 2. Statistical Tests - No Covariates

In [ ]:
from scipy.stats import f_oneway

print("="*80)
print("ANOVA + PAIRWISE TESTS (Holm-corrected) - NO COVARIATES")
print("="*80)

pairs = [('low', 'mid'), ('mid', 'high'), ('low', 'high')]

anova_results = []
pairwise_results = []

for metric in NEW_METRICS:
    print(f"\n{'='*60}")
    print(f"{metric.upper()}")
    print(f"{'='*60}")
    
    # Get groups
    groups_data = [df[df['score_bin'] == g][metric].dropna() for g in GROUP_ORDER]
    
    # ANOVA
    f_stat, p_val = f_oneway(*groups_data)
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
    print(f"\nANOVA: F = {f_stat:.3f}, p = {p_val:.4f} {sig}")
    anova_results.append({'metric': metric, 'F': f_stat, 'p': p_val})
    
    # Pairwise t-tests
    print("\nPairwise t-tests (Holm-corrected):")
    raw_pvals = []
    t_stats = []
    for g1, g2 in pairs:
        v1 = df[df['score_bin'] == g1][metric].dropna()
        v2 = df[df['score_bin'] == g2][metric].dropna()
        t, p = stats.ttest_ind(v1, v2)
        raw_pvals.append(p)
        t_stats.append(t)
    
    reject, corrected, _, _ = multipletests(raw_pvals, method='holm')
    for i, (g1, g2) in enumerate(pairs):
        sig = "***" if corrected[i] < 0.001 else "**" if corrected[i] < 0.01 else "*" if corrected[i] < 0.05 else ""
        print(f"  {g1} vs {g2}: t={t_stats[i]:>7.2f}, p_raw={raw_pvals[i]:.4f}, p_holm={corrected[i]:.4f} {sig}")
        pairwise_results.append({
            'metric': metric, 'g1': g1, 'g2': g2, 
            't': t_stats[i], 'p_raw': raw_pvals[i], 'p_holm': corrected[i]
        })

df_anova = pd.DataFrame(anova_results)
df_pairwise = pd.DataFrame(pairwise_results)

## 3. Statistical Tests - With Length Covariate

In [ ]:
print("="*80)
print("OLS REGRESSION WITH LENGTH COVARIATE")
print("="*80)

# Prepare data
df_reg = df.copy()
df_reg['score_bin'] = pd.Categorical(df_reg['score_bin'], categories=['low', 'mid', 'high'], ordered=True)
df_reg['token_count_z'] = (df_reg['token_count'] - df_reg['token_count'].mean()) / df_reg['token_count'].std()

regression_results = []
ols_models = {}

for metric in NEW_METRICS:
    print(f"\n{'='*60}")
    print(f"OLS: {metric} ~ C(score_bin) + token_count_z")
    print(f"{'='*60}")
    
    # Fit model
    formula = f'{metric} ~ C(score_bin) + token_count_z'
    model = smf.ols(formula, data=df_reg.dropna(subset=[metric])).fit()
    ols_models[metric] = model
    
    print(model.summary().tables[1])
    print(f"\nR-squared: {model.rsquared:.4f}")
    
    # Standardized beta for token_count_z
    y = df_reg[metric].dropna()
    y_std = y.std()
    beta_token = model.params.get('token_count_z', 0)
    std_beta = beta_token / y_std  # Since token_count_z is already standardized
    print(f"Standardized beta (token_count_z): {std_beta:.4f}")
    
    # Store results
    regression_results.append({
        'metric': metric,
        'r_squared': model.rsquared,
        'beta_mid': model.params.get('C(score_bin)[T.mid]', np.nan),
        'p_mid': model.pvalues.get('C(score_bin)[T.mid]', np.nan),
        'beta_high': model.params.get('C(score_bin)[T.high]', np.nan),
        'p_high': model.pvalues.get('C(score_bin)[T.high]', np.nan),
        'beta_token': beta_token,
        'p_token': model.pvalues.get('token_count_z', np.nan),
        'std_beta_token': std_beta,
    })

df_regression = pd.DataFrame(regression_results)

In [ ]:
# Summary table of regression results
print("\n" + "="*80)
print("REGRESSION SUMMARY TABLE")
print("="*80)

print(f"\n{'Metric':<15} {'R²':>8} {'β_mid':>10} {'p_mid':>10} {'β_high':>10} {'p_high':>10} {'β_token':>10} {'std_β':>8}")
print("-"*90)
for _, row in df_regression.iterrows():
    sig_mid = "*" if row['p_mid'] < 0.05 else ""
    sig_high = "*" if row['p_high'] < 0.05 else ""
    sig_tok = "*" if row['p_token'] < 0.05 else ""
    print(f"{row['metric']:<15} {row['r_squared']:>8.4f} {row['beta_mid']:>9.3f}{sig_mid} {row['p_mid']:>10.4f} {row['beta_high']:>9.3f}{sig_high} {row['p_high']:>10.4f} {row['beta_token']:>9.3f}{sig_tok} {row['std_beta_token']:>8.3f}")

## 4. Visualizations

In [ ]:
# Violin plots for all new metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metric in enumerate(NEW_METRICS):
    ax = axes[i]
    sns.violinplot(data=df, x='score_bin', y=metric, order=GROUP_ORDER,
                   palette=COLORS, ax=ax, inner='box')
    ax.set_xlabel('Score Bin', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')

plt.suptitle('Curve Shape Metrics by Score Bin', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots vs token_count for key metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, metric in enumerate(NEW_METRICS):
    ax = axes[i]
    for group in GROUP_ORDER:
        g_df = df[df['score_bin'] == group]
        ax.scatter(g_df['token_count'], g_df[metric], 
                   c=COLORS[group], label=group, alpha=0.6, s=40)
    
    # Add overall regression line
    valid = df.dropna(subset=[metric])
    z = np.polyfit(valid['token_count'], valid[metric], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid['token_count'].min(), valid['token_count'].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', alpha=0.5, linewidth=2)
    
    ax.set_xlabel('Token Count', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} vs Token Count', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Curve Metrics vs Essay Length', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of metrics
all_metrics = ['half_life', 'delta_max'] + NEW_METRICS
corr_matrix = df[all_metrics].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: All Curve Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparing effect sizes across metrics
# Use Cohen's d for low vs high comparison

def cohens_d(g1, g2):
    n1, n2 = len(g1), len(g2)
    var1, var2 = g1.var(), g2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (g1.mean() - g2.mean()) / pooled_std

effect_sizes = []
for metric in ['half_life', 'delta_max'] + NEW_METRICS:
    low_vals = df[df['score_bin'] == 'low'][metric].dropna()
    high_vals = df[df['score_bin'] == 'high'][metric].dropna()
    d = cohens_d(low_vals, high_vals)
    effect_sizes.append({'metric': metric, 'cohens_d': d})

df_effects = pd.DataFrame(effect_sizes)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#3498db' if d > 0 else '#e74c3c' for d in df_effects['cohens_d']]
bars = ax.barh(df_effects['metric'], df_effects['cohens_d'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.axvline(x=0.2, color='gray', linestyle='--', alpha=0.5, label='Small (0.2)')
ax.axvline(x=-0.2, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Medium (0.5)')
ax.axvline(x=-0.5, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel("Cohen's d (low - high)", fontsize=12)
ax.set_title("Effect Sizes: Low vs High Score Essays", fontsize=14, fontweight='bold')
ax.legend(loc='lower right')

# Add value labels
for bar, d in zip(bars, df_effects['cohens_d']):
    ax.text(d + 0.02 if d > 0 else d - 0.02, bar.get_y() + bar.get_height()/2,
            f'{d:.2f}', va='center', ha='left' if d > 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nEffect sizes (Cohen's d) for low vs high:")
print(df_effects.to_string(index=False))

## 5. Save Results

In [ ]:
# Save updated results with new metrics
output_dir = RESULTS_DIR

# Essay-level with new metrics
df.to_csv(output_dir / 'essay_level_results_with_curve_metrics.csv', index=False)

# ANOVA summary
df_anova.to_csv(output_dir / 'anova_curve_metrics.csv', index=False)

# Pairwise tests
df_pairwise.to_csv(output_dir / 'pairwise_curve_metrics.csv', index=False)

# Regression summary
df_regression.to_csv(output_dir / 'regression_curve_metrics.csv', index=False)

# Full regression outputs
with open(output_dir / 'regression_full_curve_metrics.txt', 'w') as f:
    for metric, model in ols_models.items():
        f.write(f"{'='*60}\n")
        f.write(f"{metric}\n")
        f.write(f"{'='*60}\n")
        f.write(model.summary().as_text())
        f.write("\n\n")

# Effect sizes
df_effects.to_csv(output_dir / 'effect_sizes_curve_metrics.csv', index=False)

print(f"Saved to {output_dir}/")
print(f"  - essay_level_results_with_curve_metrics.csv")
print(f"  - anova_curve_metrics.csv")
print(f"  - pairwise_curve_metrics.csv")
print(f"  - regression_curve_metrics.csv")
print(f"  - regression_full_curve_metrics.txt")
print(f"  - effect_sizes_curve_metrics.csv")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY OF KEY FINDINGS")
print("="*80)

print("\nMetrics with significant (p < 0.05) ANOVA:")
sig_anova = df_anova[df_anova['p'] < 0.05]
if len(sig_anova) > 0:
    for _, row in sig_anova.iterrows():
        print(f"  - {row['metric']}: F={row['F']:.3f}, p={row['p']:.4f}")
else:
    print("  None")

print("\nMetrics with significant score_bin effects (controlling for length):")
for _, row in df_regression.iterrows():
    sigs = []
    if row['p_mid'] < 0.05:
        sigs.append(f"mid: β={row['beta_mid']:.3f}")
    if row['p_high'] < 0.05:
        sigs.append(f"high: β={row['beta_high']:.3f}")
    if sigs:
        print(f"  - {row['metric']}: {', '.join(sigs)}")

print("\nLargest effect sizes (|d| > 0.3):")
large_effects = df_effects[abs(df_effects['cohens_d']) > 0.3].sort_values('cohens_d', key=abs, ascending=False)
if len(large_effects) > 0:
    for _, row in large_effects.iterrows():
        direction = "low > high" if row['cohens_d'] > 0 else "high > low"
        print(f"  - {row['metric']}: d={row['cohens_d']:.3f} ({direction})")
else:
    print("  None")